# Brain Tumor Segmentation — 2D U-Net Training Pipeline

This notebook runs the full ML pipeline: **split → load → train → evaluate**.

**Dataset:** Download [BraTS 2020](https://www.kaggle.com/datasets/awsaf49/brats2020-training-data) from Kaggle. Extract so that `volume_XXX_slice_YYY.h5` files are in a data directory.

**Outputs (saved locally, not pushed to Git):**
- `splits/` — train_ids.txt, val_ids.txt, test_ids.txt
- `best_model.h5` — trained 2D U-Net

## 1. Configuration

In [ ]:
# Paths — edit these for your environment
DATA_DIR = r"C:\Users\Sravani\Downloads\archive\BraTS2020_training_data\content\data"  # BraTS slice .h5 files
SPLITS_DIR = "splits"  # Where to write train/val/test_ids.txt
MODEL_PATH = "best_model.h5"  # Where to save the trained model

## 2. Split dataset (70 / 15 / 15)

In [ ]:
from pathlib import Path
import sys
for p in [Path.cwd(), Path.cwd().parent]:
    if (p / "split_brats_dataset.py").exists():
        sys.path.insert(0, str(p))
        break

from split_brats_dataset import get_volume_ids_from_directory, split_volumes

Path(SPLITS_DIR).mkdir(parents=True, exist_ok=True)
volume_ids = get_volume_ids_from_directory(DATA_DIR)
print(f"Found {len(volume_ids)} volumes")

train_ids, val_ids, test_ids = split_volumes(volume_ids, random_state=42)
print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

Path(SPLITS_DIR).mkdir(parents=True, exist_ok=True)
Path(SPLITS_DIR, "train_ids.txt").write_text("\n".join(train_ids) + "\n")
Path(SPLITS_DIR, "val_ids.txt").write_text("\n".join(val_ids) + "\n")
Path(SPLITS_DIR, "test_ids.txt").write_text("\n".join(test_ids) + "\n")
print(f"Saved splits to {SPLITS_DIR}/")

## 3. Load training and validation data

In [ ]:
from training.dataset_loader_2d import load_split_2d

print("Loading TRAIN...")
X_train, Y_train = load_split_2d(Path(SPLITS_DIR) / "train_ids.txt", DATA_DIR, random_state=42)
print("Loading VAL...")
X_val, Y_val = load_split_2d(Path(SPLITS_DIR) / "val_ids.txt", DATA_DIR, random_state=123)

X_train = X_train.astype("float32")
Y_train = Y_train.astype("float32")
X_val = X_val.astype("float32")
Y_val = Y_val.astype("float32")
print(f"X_train: {X_train.shape}, Y_train: {Y_train.shape}")
print(f"X_val:   {X_val.shape},   Y_val:   {Y_val.shape}")

## 4. Build and train 2D U-Net

In [ ]:
import tensorflow as tf
from training.unet2d_model import build_unet_2d

model = build_unet_2d(
    input_shape=X_train.shape[1:],
    base_filters=32,
    num_classes=Y_train.shape[-1],
    learning_rate=1e-4,
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_dice_coefficient_multiclass", mode="max", patience=5, restore_best_weights=True
)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    MODEL_PATH, monitor="val_dice_coefficient_multiclass", mode="max", save_best_only=True
)

model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=30,
    batch_size=8,
    callbacks=[early_stopping, checkpoint],
)
print(f"Best model saved to {MODEL_PATH}")

## 5. Evaluate on test set (per-volume Dice)

In [ ]:
import numpy as np
from training.dataset_loader_2d import _parse_volume_number, _load_slices_for_volume, _read_volume_ids

def dice_per_volume_multiclass(y_true, y_pred, eps=1e-6):
    axes = tuple(range(y_true.ndim - 1))
    intersection = np.sum(y_true * y_pred, axis=axes)
    denominator = np.sum(y_true, axis=axes) + np.sum(y_pred, axis=axes)
    return float(np.mean((2.0 * intersection + eps) / (denominator + eps)))

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
volume_ids = _read_volume_ids(Path(SPLITS_DIR) / "test_ids.txt")
dice_scores = []

for vol_id in volume_ids:
    vol_num = _parse_volume_number(vol_id)
    X_slices, Y_slices = _load_slices_for_volume(Path(DATA_DIR), vol_num)
    pred_probs = model.predict(X_slices, batch_size=8, verbose=0)
    num_classes = pred_probs.shape[-1]
    gt_bin = (Y_slices > 0.5).astype("float32")
    gt_labels = np.argmax(gt_bin, axis=-1)
    gt_one_hot = tf.one_hot(gt_labels, depth=num_classes).numpy().astype("float32")
    pred_labels = np.argmax(pred_probs, axis=-1)
    pred_one_hot = tf.one_hot(pred_labels, depth=num_classes).numpy().astype("float32")
    dice = dice_per_volume_multiclass(gt_one_hot, pred_one_hot)
    dice_scores.append(dice)
    print(f"{vol_id}: Dice = {dice:.4f}")

dice_arr = np.array(dice_scores)
print(f"\nMean Dice: {dice_arr.mean():.4f}")
print(f"Std Dice:  {dice_arr.std():.4f}")